**MIT805 Big Data Semester Project - Part 1**

Aeolus MFDD Multi-Modal Flight Delay

Group 4- u23056534 & u20426799

3 September 2026

In [1]:
!pip install kagglehub

import kagglehub

path = kagglehub.dataset_download("flnny123/mfddmulti-modal-flight-delay-dataset")
print("Dataset downloaded to:", path)

100%|██████████| 5.22G/5.22G [01:15<00:00, 74.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4


In [2]:
#Inspecting the datasets and folders
path = "/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4"
!ls -lh $path
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus

#Inspect each folder of the dataset
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_chain
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_network
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_Tab

print(path)
!ls -lh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4



total 4.0K
drwxr-xr-x 5 root root 4.0K Sep  3 06:40 Aeolus
total 12K
drwxr-xr-x 11 root root 4.0K Sep  3 06:39 Flight_chain
drwxr-xr-x 11 root root 4.0K Sep  3 06:42 Flight_network
drwxr-xr-x  2 root root 4.0K Sep  3 06:38 Flight_Tab
total 12K
drwxr-xr-x 11 root root 4.0K Sep  3 06:39 Flight_chain
drwxr-xr-x 11 root root 4.0K Sep  3 06:42 Flight_network
drwxr-xr-x  2 root root 4.0K Sep  3 06:38 Flight_Tab
ls: cannot access '/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_chain': No such file or directory
ls: cannot access '/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_network': No such file or directory
ls: cannot access '/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Flight_Tab': No such file or directory
/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4
total 4.0K
drwxr-xr-x 5 root root 4.0K Sep  3 06:40 Aeo

In [3]:
#uncompressed files size:
!du -sh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4
#Tabular CSV data size:
!du -sh /root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab


42G	/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4
15G	/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayAnalysis") \
    .getOrCreate()

In [5]:
flight_tab_path = path + "/Aeolus/Flight_Tab"     ## Define the path to the folder containing the flight tabular data

In [6]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(flight_tab_path)

In [7]:
## Create Parquet file to store processed info to save time

#df.write.mode("overwrite").parquet("/content/drive/MyDrive/flight_delay_checkpoint.parquet")

In [8]:
## Check the number of missing values in each column

from pyspark.sql import functions as F

missing_values = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show(vertical=True)

-RECORD 0--------------------
 FL_DATE             | 0     
 OP_CARRIER          | 0     
 OP_CARRIER_FL_NUM   | 0     
 ORIGIN              | 0     
 DEST                | 0     
 CRS_DEP_TIME        | 0     
 DEP_TIME            | 0     
 DEP_DELAY           | 0     
 TAXI_OUT            | 0     
 WHEELS_OFF          | 0     
 WHEELS_ON           | 0     
 TAXI_IN             | 0     
 CRS_ARR_TIME        | 0     
 ARR_TIME            | 0     
 ARR_DELAY           | 0     
 CRS_ELAPSED_TIME    | 0     
 ACTUAL_ELAPSED_TIME | 0     
 AIR_TIME            | 0     
 FLIGHTS             | 0     
 MONTH               | 0     
 DAY_OF_MONTH        | 0     
 DAY_OF_WEEK         | 0     
 ORIGIN_INDEX        | 0     
 DEST_INDEX          | 0     
 O_TEMP              | 13474 
 O_PRCP              | 13474 
 O_WSPD              | 13474 
 D_TEMP              | 15046 
 D_PRCP              | 15046 
 D_WSPD              | 15046 
 O_LATITUDE          | 0     
 O_LONGITUDE         | 0     
 D_LATITUD

In [9]:
## Calculate the percentage of missing values in each column

count_total_rows = df.count()

missing_percentage = df.select([
    (
        F.sum(F.col(c).isNull().cast("int"))
        / count_total_rows * 100
    ).alias(c)
    for c in df.columns
])

missing_percentage.show(vertical=True)

-RECORD 0-----------------------------------
 FL_DATE             | 0.0                  
 OP_CARRIER          | 0.0                  
 OP_CARRIER_FL_NUM   | 0.0                  
 ORIGIN              | 0.0                  
 DEST                | 0.0                  
 CRS_DEP_TIME        | 0.0                  
 DEP_TIME            | 0.0                  
 DEP_DELAY           | 0.0                  
 TAXI_OUT            | 0.0                  
 WHEELS_OFF          | 0.0                  
 WHEELS_ON           | 0.0                  
 TAXI_IN             | 0.0                  
 CRS_ARR_TIME        | 0.0                  
 ARR_TIME            | 0.0                  
 ARR_DELAY           | 0.0                  
 CRS_ELAPSED_TIME    | 0.0                  
 ACTUAL_ELAPSED_TIME | 0.0                  
 AIR_TIME            | 0.0                  
 FLIGHTS             | 0.0                  
 MONTH               | 0.0                  
 DAY_OF_MONTH        | 0.0                  
 DAY_OF_WE

In [10]:
!rm -rf /content/drive

Restart from here

In [11]:
## Load Spark

#from pyspark.sql import SparkSession

#Spark = SparkSession.builder.getOrCreate()

## Load Parquet file
#from google.colab import drive
#drive.mount('/content/drive')

#df = Spark.read.parquet(
#    "/content/drive/MyDrive/flight_delay_checkpoint.parquet"
#)

In [12]:
#import os

#for root, dirs, files in os.walk("/content/drive"):
#    for name in dirs:
#        if "flight" in name.lower() or "parquet" in name.lower():
#            print(os.path.join(root, name))

In [13]:
## Count Duplicates

Rows_count = df.count()
Unique_count = df.dropDuplicates().count()

print("Duplicates Percentage (%):", (Rows_count-Unique_count)/Rows_count*100)

Duplicates Percentage (%): 0.0


In [14]:
## Check for Outliers in Numerical Data -> Manual inspection on selected variables

df.select(
    "DEP_DELAY",
    "ARR_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "O_TEMP",
    "O_PRCP",
    "O_WSPD",
    "D_TEMP",
    "D_PRCP",
    "D_WSPD"
).summary().show()

+-------+------------------+-----------------+-----------------+-----------------+------------------+-------------------+-----------------+------------------+-------------------+------------------+
|summary|         DEP_DELAY|        ARR_DELAY|         TAXI_OUT|          TAXI_IN|            O_TEMP|             O_PRCP|           O_WSPD|            D_TEMP|             D_PRCP|            D_WSPD|
+-------+------------------+-----------------+-----------------+-----------------+------------------+-------------------+-----------------+------------------+-------------------+------------------+
|  count|          54674003|         54674003|         54674003|         54674003|          54660529|           54660529|         54660529|          54658957|           54658957|          54658957|
|   mean|10.092505591734339|4.477098521577065|16.91077375475873|7.722854827366491|16.144790426011042|0.20139118668242437|12.46590276612538| 16.81172826935298|0.22081131917684862|13.037415218827178|
| stddev|4

In [15]:
## Calculate delay frequency for EDA

from pyspark.sql.functions import when, col

delay_threshold = 15   # Set to 15 minutes delay; https://transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ&utm_source=chatgpt.com

df = df.withColumn("Delayed_flights", when(col("DEP_DELAY") >= delay_threshold, 1).otherwise(0))

df.groupby("Delayed_flights").count()

df.groupby("Delayed_flights") \
    .count() \
    .withColumn("Percentage", col("count") / df.count() * 100) \
    .show()

+---------------+--------+-----------------+
|Delayed_flights|   count|       Percentage|
+---------------+--------+-----------------+
|              1| 9969616|18.23465532604225|
|              0|44704387|81.76534467395776|
+---------------+--------+-----------------+



In [16]:
## Determine delay by month

monthly_delay = (
    df.groupBy("MONTH")
      .agg(F.avg("ARR_DELAY").alias("average_arrival_delay(Minutes)"))
      .orderBy("MONTH")
)

monthly_delay.show()

+-----+------------------------------+
|MONTH|average_arrival_delay(Minutes)|
+-----+------------------------------+
|    1|            3.5945581628344065|
|    2|              2.60112299348055|
|    3|            2.3940526120937005|
|    4|            3.8935935486233073|
|    5|             5.350208758470634|
|    6|             9.630656033369421|
|    7|            10.438491582865375|
|    8|             7.539011230293804|
|    9|            0.9131365928921117|
|   10|             1.015696132745295|
|   11|           0.00738276331504191|
|   12|             5.435417082693327|
+-----+------------------------------+



In [17]:
## Determine Carrier Contribution

delay_by_carrier = df.groupBy("OP_CARRIER") \
    .agg(
        F.count("*").alias("Total_Flights"),
        F.sum("Delayed_flights").alias("Delayed_Flights")
    )

delay_by_carrier = delay_by_carrier.withColumn(
    "Delayed_Percentage",
    F.col("Delayed_Flights") / F.col("Total_Flights") * 100
)

delay_by_carrier.orderBy(
    F.desc("Delayed_Percentage")
).show()

+----------+-------------+---------------+------------------+
|OP_CARRIER|Total_Flights|Delayed_Flights|Delayed_Percentage|
+----------+-------------+---------------+------------------+
|        B6|      2236608|         588850| 26.32781426159613|
|        F9|      1176549|         301744|  25.6465306587316|
|        G4|       303824|          69106| 22.74540523460951|
|        VX|       155092|          34993|22.562736956129264|
|        WN|     11133643|        2460739|22.101831359241537|
|        NK|      1696082|         365310| 21.53846335259734|
|        AA|      7518473|        1440670| 19.16173669839607|
|        UA|      5083571|         918186| 18.06183094521548|
|        EV|      1178841|         211839|17.970107928041188|
|        YV|       818626|         143496|17.528883763770025|
|        OH|      1538083|         260583|16.942063594747488|
|        MQ|      1747132|         274790|15.728061760645446|
|        OO|      6044308|         922459|15.261614729097195|
|       

In [18]:
## Airport delay at origin
from pyspark.sql import functions as F

origin_delay = (
    df.groupBy("ORIGIN")
      .agg(
          F.avg("DEP_DELAY").alias("avg_dep_delay"),
          F.expr("percentile(DEP_DELAY, 0.5)").alias("median_dep_delay"),
          (F.sum(F.when(df.DEP_DELAY > 15, 1).otherwise(0)) / F.count("*")).alias("pct_delay_15"),
          (F.sum(F.when(df.DEP_DELAY > 60, 1).otherwise(0)) / F.count("*")).alias("pct_delay_60")
      )
      .orderBy(F.desc("avg_dep_delay"))
)

origin_delay.show(20, truncate=False)


+------+------------------+----------------+-------------------+-------------------+
|ORIGIN|avg_dep_delay     |median_dep_delay|pct_delay_15       |pct_delay_60       |
+------+------------------+----------------+-------------------+-------------------+
|STC   |40.5              |5.0             |0.3333333333333333 |0.16666666666666666|
|PPG   |34.4327990135635  |-1.0            |0.21085080147965474|0.0690505548705302 |
|MMH   |27.742376445846478|-1.0            |0.28391167192429023|0.12407991587802314|
|OTH   |25.049845307665866|-2.0            |0.2643520110003438 |0.12169130285321417|
|UST   |24.77734375       |-1.0            |0.25390625         |0.140625           |
|HYA   |23.410516605166052|-1.0            |0.2693726937269373 |0.12638376383763839|
|ILG   |22.955696202531644|0.5             |0.36075949367088606|0.1518987341772152 |
|ASE   |20.620154882832143|-4.0            |0.24914009856180228|0.12533440611485466|
|MVY   |20.213628620102213|-3.0            |0.2517887563884157 |0

In [19]:
## Airport delay at destination
dest_delay = (
    df.groupBy("DEST")
      .agg(
          F.avg("ARR_DELAY").alias("avg_arr_delay"),
          F.expr("percentile(ARR_DELAY, 0.5)").alias("median_arr_delay"),
          (F.sum(F.when(df.ARR_DELAY > 15, 1).otherwise(0)) / F.count("*")).alias("pct_delay_15"),
          (F.sum(F.when(df.ARR_DELAY > 60, 1).otherwise(0)) / F.count("*")).alias("pct_delay_60")
      )
      .orderBy(F.desc("avg_arr_delay"))
)

dest_delay.show(20, truncate=False)


+----+------------------+----------------+-------------------+-------------------+
|DEST|avg_arr_delay     |median_arr_delay|pct_delay_15       |pct_delay_60       |
+----+------------------+----------------+-------------------+-------------------+
|STC |39.0              |15.0            |0.42857142857142855|0.14285714285714285|
|PPG |18.064356435643564|1.0             |0.22400990099009901|0.04702970297029703|
|BQN |17.4911291270121  |1.0             |0.32264128774527084|0.11949242157208319|
|PSE |16.600096169257895|0.0             |0.3136720628305818 |0.12437890687610194|
|ASE |15.632954775907525|-4.0            |0.2565808152504563 |0.10567836138714257|
|ILG |15.234177215189874|4.0             |0.25949367088607594|0.10126582278481013|
|MMH |14.517782426778243|-5.0            |0.24686192468619247|0.09832635983263599|
|PBG |13.66406429391504 |-7.0            |0.18874856486796784|0.07944890929965556|
|SWF |12.629647132955261|-3.0            |0.2526780088216761 |0.10034656584751103|
|UST

In [20]:
#airport congestion
df = df.withColumn("DEP_HOUR", F.hour("DEP_TIME"))
congestion = (
    df.groupBy("ORIGIN", "DEP_HOUR")
      .count()
      .groupBy("ORIGIN")
      .agg(F.avg("count").alias("avg_flights_per_hour"))
      .orderBy(F.desc("avg_flights_per_hour"))
)

congestion.show(20, truncate=False)


+------+--------------------+
|ORIGIN|avg_flights_per_hour|
+------+--------------------+
|ATL   |125944.58333333333  |
|ORD   |96262.41666666667   |
|DFW   |93549.0             |
|DEN   |89905.95833333333   |
|LAX   |70663.54166666667   |
|CLT   |66782.79166666667   |
|PHX   |59787.291666666664  |
|LAS   |57763.958333333336  |
|SEA   |53453.833333333336  |
|SFO   |51699.041666666664  |
|MCO   |49767.958333333336  |
|IAH   |48580.5             |
|DTW   |48425.75            |
|MSP   |47641.958333333336  |
|LGA   |46906.5             |
|BOS   |45400.5             |
|EWR   |42506.25            |
|SLC   |40478.208333333336  |
|DCA   |39679.75            |
|JFK   |39390.791666666664  |
+------+--------------------+
only showing top 20 rows


In [21]:
route_delay = (
    df.groupBy("ORIGIN", "DEST")
      .agg(F.avg("ARR_DELAY").alias("average_arrival_delay(Minutes)"),
           F.sum(F.when(F.col("ARR_DELAY") > 0, 1).otherwise(0))
            .alias("delayed_flights"))
      .orderBy(F.desc("average_arrival_delay(Minutes)"))
)

route_delay.show(20, truncate=False)


+------+----+------------------------------+---------------+
|ORIGIN|DEST|average_arrival_delay(Minutes)|delayed_flights|
+------+----+------------------------------+---------------+
|CAK   |TYS |1237.0                        |1              |
|BUR   |FLL |922.0                         |1              |
|MDT   |HPN |798.0                         |1              |
|JFK   |LGA |755.0                         |1              |
|VPS   |SRQ |744.0                         |1              |
|CVG   |CLE |413.0                         |2              |
|FLL   |CHA |385.0                         |1              |
|SDF   |SLC |365.0                         |1              |
|JNU   |FAI |334.0                         |1              |
|CLL   |MIA |262.0                         |1              |
|ICT   |DAY |210.0                         |1              |
|AUS   |SAF |206.25                        |5              |
|CHA   |MSP |206.0                         |1              |
|CLE   |ERI |177.0      

In [22]:
from pyspark.sql import functions as F

df_perf = df.select("OP_CARRIER", "ARR_DELAY")

df_perf = df_perf.withColumn(
    "on_time",
    F.when(df_perf.ARR_DELAY <= 0, 1).otherwise(0)
)

on_time_rank = (
    df_perf.groupBy("OP_CARRIER")
           .agg(
               (F.avg("on_time")*100).alias("on_time_rate"),
               F.count("*").alias("num_flights")
           )
           .orderBy(F.desc("on_time_rate"))
)

on_time_rank.show(20)



+----------+------------------+-----------+
|OP_CARRIER|      on_time_rate|num_flights|
+----------+------------------+-----------+
|        9E| 75.43822941964767|    1558088|
|        DL| 71.91681098815621|    7821658|
|        YX| 71.43630985459562|    1995607|
|        OO| 68.28655985102017|    6044308|
|        UA| 67.08148661639623|    5083571|
|        EV| 66.22428300339061|    1178841|
|        OH|  65.5824815695902|    1538083|
|        YV| 65.39628597186017|     818626|
|        MQ| 65.36930237669506|    1747132|
|        QX| 65.18649354331825|     169282|
|        AS| 63.69636271533158|    1859574|
|        AA| 63.38042312581291|    7518473|
|        NK| 62.85427237598182|    1696082|
|        WN|  62.5488710209228|   11133643|
|        HA|60.904717338433265|     638962|
|        B6| 59.38233253212006|    2236608|
|        F9| 59.00672220196523|    1176549|
|        G4| 57.74099478645531|     303824|
|        VX|55.120831506460675|     155092|
+----------+------------------+-

In [23]:
#import os

#file_path = "/content/drive/MyDrive/flight_delay_checkpoint.parquet"
#size_mb = os.path.getsize(file_path) / (1024 * 1024)

#print(f"Single parquet file size: {size_mb:.2f} MB")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/flight_delay_checkpoint.parquet'

In [ ]:
#import os

#def folder_size_mb(path):
#    total = 0
#    for root, dirs, files in os.walk(path):
#        for f in files:
#            total += os.path.getsize(os.path.join(root, f))
#    return total / (1024 * 1024)

#print(folder_size_mb("/tmp/df_size_check"))


In [24]:
import os

def folder_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 * 1024)

print(folder_size_mb("/root/.cache/kagglehub/datasets/flnny123/mfddmulti-modal-flight-delay-dataset/versions/4/Aeolus/Flight_Tab"))


14492.384121894836


In [25]:
import os

def folder_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 * 1024)

print(folder_size_mb("/content/drive/MyDrive/flight_delay_checkpoint.parquet"))

0.0


In [26]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Create day_of_week column
# ---------------------------------------------------------
df_dow = df.withColumn("day_of_week", F.dayofweek("FL_DATE"))

# ---------------------------------------------------------
# 2. Create delay indicator (≥10 minutes)
# ---------------------------------------------------------
df_dow = df_dow.withColumn(
    "delayed_10",
    F.when(F.col("ARR_DELAY") >= 10, 1).otherwise(0)
)

# ---------------------------------------------------------
# 3. Create on-time indicator (ARR_DELAY ≤ 0)
# ---------------------------------------------------------
df_dow = df_dow.withColumn(
    "on_time",
    F.when(F.col("ARR_DELAY") <= 0, 1).otherwise(0)
)

# ---------------------------------------------------------
# 4. Compute day-of-week summary table
# ---------------------------------------------------------
dow_summary = (
    df_dow.groupBy("day_of_week")
          .agg(
              F.avg("ARR_DELAY").alias("avg_arr_delay"),
              (F.avg("delayed_10") * 100).alias("delay_rate_percent"),
              (F.avg("on_time") * 100).alias("on_time_rate_percent"),
              F.count("*").alias("num_flights")
          )
          .orderBy("day_of_week")
)

dow_summary.show()

# ---------------------------------------------------------
# 5. Optional: Add weekday names for readability
# ---------------------------------------------------------
dow_summary_named = dow_summary.withColumn(
    "weekday",
    F.when(F.col("day_of_week") == 1, "Sunday")
     .when(F.col("day_of_week") == 2, "Monday")
     .when(F.col("day_of_week") == 3, "Tuesday")
     .when(F.col("day_of_week") == 4, "Wednesday")
     .when(F.col("day_of_week") == 5, "Thursday")
     .when(F.col("day_of_week") == 6, "Friday")
     .otherwise("Saturday")
)

dow_summary_named.orderBy("day_of_week").show()


+-----------+------------------+------------------+--------------------+-----------+
|day_of_week|     avg_arr_delay|delay_rate_percent|on_time_rate_percent|num_flights|
+-----------+------------------+------------------+--------------------+-----------+
|          1|5.1063596997136464|23.050509834247148|   65.28865313590991|    7867557|
|          2|5.1268760383032665|22.861689377366215|     65.411027166011|    8142491|
|          3| 2.524915919064818|20.358098049115174|   68.40020971688222|    7724700|
|          4|2.9472510000236083| 20.94030703316578|   67.48043577112932|    7793816|
|          5| 5.774830314820295| 23.98624109102613|   63.70766519246843|    8093665|
|          6| 6.428813242627223|24.681722859555386|   62.97945362720908|    8145574|
|          7|3.0813347426949695| 20.79907908835539|     68.069401407431|    6906200|
+-----------+------------------+------------------+--------------------+-----------+

+-----------+------------------+------------------+-------------

In [27]:
df = df.withColumn("YEAR", F.year("FL_DATE"))

yearly_delay = (
    df.groupBy("YEAR")
      .agg(
          F.avg("ARR_DELAY").alias("average_arrival_delay"),
          F.count("*").alias("number_of_flights")
      )
      .orderBy("YEAR")
)

yearly_delay.show()

+----+---------------------+-----------------+
|YEAR|average_arrival_delay|number_of_flights|
+----+---------------------+-----------------+
|2016|    3.519447951033471|          5537987|
|2017|    4.321907497159189|          5575872|
|2018|    4.997999954772127|          6986842|
|2019|    5.378661059531318|          7161827|
|2020|   -5.107839560899805|          4312091|
|2021|   2.9493552961551277|          5755666|
|2022|    6.934429171599035|          6413416|
|2023|    6.568991526697697|          6645461|
|2024|    7.108082925248228|          6284841|
+----+---------------------+-----------------+

